# DS317 - KHAI PHÁ DỮ LIỆU TRONG DOANH NGHIỆP (UIT)
## BÀI TẬP VỀ NHÀ BT5: GOM NHÓM VÀ XÁC ĐỊNH NGOẠI LAI (CHƯƠNG 8)
**ĐỀ TÀI: DỰ BÁO THỜI ĐIỂM CẬP CẢNG CỦA TÀU BIỂN THAY THẾ ETA TỰ KHAI BÁO**  
**Nhóm thực hiện: Nhóm 03 (Lớp DS317.P11 / DS317.R11)**  
*Môi trường thực thi: Tương thích hoàn toàn cả Kaggle Cloud (Dataset NOAA AIS 2025) và Local Workspace*

---

### MỤC TIÊU VẬN DỤNG NỘI DUNG CHƯƠNG 8:
1. **Khai thác dữ liệu không nhãn (Unlabeled Data Mining):** Ứng dụng triết lý *"Hiểu bình thường để tìm bất thường"* trên dữ liệu động học hàng hải NOAA AIS 2025.
2. **Khung 5 tiêu chí lựa chọn thuật toán (Slide 28 & 38):**
   - Đánh giá theo: Thời gian tính toán & độ phức tạp $\mathcal{O}$, loại mô hình, ý tưởng cốt lõi, ưu điểm, nhược điểm.
   - Cài đặt $\ge 3$ phương pháp gom nhóm: **K-means (Baseline)**, **GMM**, **Ward Hierarchical**, **DBSCAN**.
   - Cài đặt $\ge 3$ phương pháp ngoại lai: **Tukey IQR / Modified Z-score (Baseline)**, **Khoảng cách tâm cụm**, **Isolation Forest**, **Local Outlier Factor (LOF)**.
3. **Đánh giá kết hợp 4 góc độ toàn diện (Slide 41-51):**
   - **Nội bộ (Internal):** Phương pháp khuỷu tay SSE (Elbow), Silhouette Score, Calinski-Harabasz (CH), Davies-Bouldin (DB), BIC và ràng buộc tỷ trọng cụm nhỏ nhất $\ge 5\%$.
   - **Bên ngoài (External):** Đối chiếu với nhãn tự nhiên `Is_Tanker` (Tàu hàng vs Tàu dầu) qua Purity, Entropy, Adjusted Rand Index (ARI), NMI.
   - **Trực quan (Visual):** Không gian PCA 2D, Silhouette Plot từng cụm, Heatmap Z-Score hồ sơ cụm, Cây quyết định thay thế (Surrogate Decision Tree).
   - **Nghiệp vụ (Domain/Business):** Hồ sơ hóa 4 phân khúc hành vi hành hải, kiểm định độ ổn định Bootstrap ($B=30$, đo ARI), phát hiện 3 loại ngoại lai (Toàn cục, Ngữ cảnh, Tập thể), xuất cảnh báo ngoại lai kèm lý do đọc được, và đo lường độ trôi dữ liệu bằng chỉ số PSI.


In [ ]:
import os
import sys
import time
import glob
import json
import zipfile
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import (
    silhouette_score, silhouette_samples,
    calinski_harabasz_score, davies_bouldin_score,
    adjusted_rand_score, normalized_mutual_info_score
)

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Co dinh hat giong ngau nhien de tai lap 100% ket qua
SEED = 42
np.random.seed(SEED)

print("Python version :", sys.version.split()[0])
print("Pandas version :", pd.__version__)
print("NumPy version  :", np.__version__)
print("Moi truong san sang!")


In [ ]:
# Cau hinh thu muc dau ra
IS_KAGGLE = os.path.exists('/kaggle')
WORKING_DIR = '/kaggle/working' if IS_KAGGLE else '.'
OUTPUT_DIR = os.path.join(WORKING_DIR, 'Ket_Qua_Thuc_Nghiem')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Moi truong: {'KAGGLE CLOUD' if IS_KAGGLE else 'LOCAL WORKSPACE'}")
print(f"Thu muc xuat ket qua: {OUTPUT_DIR}")

def find_dataset_file():
    """Tu dong do tim tap du lieu dau vao tren Kaggle hoac Local."""
    candidates = [
        # 1. Duong dan chuan Dataset cua nhom tren Kaggle (Ban 22 thuoc tinh)
        "/kaggle/input/datasets/kienvo134/noaa-ais-2025-full-year-tin-x-l-nn-tng/02_ban_nen_tang_22_thuoc_tinh/train.parquet",
        "/kaggle/input/noaa-ais-2025-full-year-tin-x-l-nn-tng/02_ban_nen_tang_22_thuoc_tinh/train.parquet",
        # 2. Ban 27 thuoc tinh tren Kaggle
        "/kaggle/input/datasets/kienvo134/noaa-ais-2025-full-year-tin-x-l-nn-tng/03_ban_tien_xu_ly_27_thuoc_tinh/train.parquet",
        "/kaggle/input/noaa-ais-2025-full-year-tin-x-l-nn-tng/03_ban_tien_xu_ly_27_thuoc_tinh/train.parquet",
        # 3. Cache parquet cuc bo
        "./Ket_Qua_Thuc_Nghiem/ais_sample_clustering_outlier.parquet",
        "../Ket_Qua_Thuc_Nghiem/ais_sample_clustering_outlier.parquet",
        "../../Ket_Qua_Thuc_Nghiem/ais_sample_clustering_outlier.parquet",
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
            
    # Do tim de quy tat ca cac file parquet co chu 'train' hoac 'ais'
    if IS_KAGGLE:
        matches = glob.glob('/kaggle/input/**/train.parquet', recursive=True)
        if matches:
            return matches[0]
        matches = glob.glob('/kaggle/input/**/*.parquet', recursive=True)
        if matches:
            return matches[0]
    else:
        matches = glob.glob('**/*sample*.parquet', recursive=True)
        if matches:
            return matches[0]
    return None

DATA_PATH = find_dataset_file()
print(f"Tep du lieu duoc phat hien: {DATA_PATH}")


In [ ]:
if DATA_PATH is None:
    raise FileNotFoundError("Khong tim thay tep du lieu train.parquet! Vui long kiem tra Input tren Kaggle.")

print(f"Dang doc du lieu tu: {DATA_PATH}...")
df_raw = pd.read_parquet(DATA_PATH)
total_rows = len(df_raw)
print(f"Tong so ban ghi goc: {total_rows:,} dong x {df_raw.shape[1]} cot.")

# Lay mau phan tang theo Is_Tanker neu tap du lieu qua lon (de chay toi uu tren Kaggle RAM)
N_SAMPLE = 50000
if total_rows > N_SAMPLE:
    print(f"Tap du lieu lon ({total_rows:,} dong) -> Lay mau phan tang {N_SAMPLE:,} dong de toi uu hoa hieu nang...")
    df = df_raw.groupby('Is_Tanker', group_keys=False).apply(
        lambda x: x.sample(int(np.rint(N_SAMPLE * len(x) / total_rows)), random_state=SEED)
    ).reset_index(drop=True)
else:
    df = df_raw.copy()

print(f"So mau thuc nghiem: {len(df):,} dong.")

# Tinh toan/Dam bao cac bien luong giac va goc lai
if 'cog_sin' not in df.columns:
    cog_rad = np.radians(df['cog'].fillna(0.0))
    df['cog_sin'] = np.sin(cog_rad)
    df['cog_cos'] = np.cos(cog_rad)

if 'abs_delta_heading' not in df.columns:
    diff_angle = np.abs(df['cog'].fillna(0.0) - df['heading'].fillna(df['cog']).fillna(0.0))
    df['abs_delta_heading'] = np.where(diff_angle > 180.0, 360.0 - diff_angle, diff_angle)
    df['abs_delta_heading'] = np.nan_to_num(df['abs_delta_heading'], nan=0.0)

if 'v_implied' not in df.columns:
    df['v_implied'] = np.where(
        (df['Target_Hours'] > 0) & (df['Dist_To_Berth_NM'] > 0),
        (df['Dist_To_Berth_NM'] / df['Target_Hours']).clip(lower=0.5, upper=40.0),
        df['sog']
    )

if 'vessel_group' not in df.columns:
    df['vessel_group'] = np.where(df['Is_Tanker'] == 1, 'Tanker (Tau dau)', 'Cargo (Tau hang)')

# Khong gian 8 dac trung hanh hai chuyen biet
FEATURE_COLS = [
    'sog', 'Dist_To_Berth_NM', 'draft', 'Area_Proxy',
    'cog_sin', 'cog_cos', 'abs_delta_heading', 'v_implied'
]

FEATURE_LABELS_VN = {
    'sog': 'Toc do SOG (knots)',
    'Dist_To_Berth_NM': 'Cu ly den ben (NM)',
    'draft': 'Mon nuoc (m)',
    'Area_Proxy': 'Dien tich tau (m2)',
    'cog_sin': 'Huong di chuyen Sin',
    'cog_cos': 'Huong di chuyen Cos',
    'abs_delta_heading': 'Do lech huong mui (do)',
    'v_implied': 'Toc do suy dien (knots)'
}

# Chuan hoa StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[FEATURE_COLS].fillna(0.0))
joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'scaler_clustering.joblib'))

print("Khong gian dac trung:", FEATURE_COLS)
print("Kich thuoc ma tran X_scaled:", X_scaled.shape)
display(df[FEATURE_COLS + ['Is_Tanker']].describe().T[['mean', 'std', 'min', '50%', 'max']])


## 2. KHUNG 5 TIÊU CHÍ LỰA CHỌN THUẬT TOÁN (SLIDE 28 & 38)
Tuân thủ đề cương môn học DS317, nhóm đối chiếu các mô hình phân cụm và phát hiện ngoại lai theo 5 tiêu chí:
1. **Thời gian tính toán & độ phức tạp $\mathcal{O}$**
2. **Loại mô hình**
3. **Ý tưởng cốt lõi**
4. **Ưu điểm**
5. **Nhược điểm**


In [ ]:
# Lap bang 5 tieu chi thuat toan phan cum
clustering_criteria = [
    {
        'Thuat_Toan': 'K-means (Baseline)',
        'Thoi_Gian': '1967 (MacQueen) · O(N·k·d·I) · Rat nhanh',
        'Loai_Mo_Hinh': 'Phan hoach (Partition-based)',
        'Y_Tuong_Cot_Loi': 'Toi thieu hoa tong binh phuong khoang cach noi cum W(C)',
        'Uu_Diem': 'Tuyen tinh voi N, chay cuc nhanh tren Big Data, de dien giai',
        'Nhuoc_Diem': 'Chi tim cum hinh cau loi, nhay voi ngoai lai, can biet truoc k'
    },
    {
        'Thuat_Toan': 'GMM (Gaussian Mixture)',
        'Thoi_Gian': '1977 (Dempster et al.) · O(N·k·d^2·I) · Trung binh',
        'Loai_Mo_Hinh': 'Xac suat/Hon hop (Probabilistic)',
        'Y_Tuong_Cot_Loi': 'Cuc dai hoa hop ly EM voi ma tran hiep phuong sai mem',
        'Uu_Diem': 'Tim duoc cum elip linh hoat, cho xac suat mem tung cum, tinh BIC',
        'Nhuoc_Diem': 'Nhay voi diem khoi tao, nguy co cuc tieu cuc bo, ton RAM'
    },
    {
        'Thuat_Toan': 'Ward Hierarchical',
        'Thoi_Gian': '1963 (Ward) · O(N^2 log N) · Cham tren tap lon',
        'Loai_Mo_Hinh': 'Phan cap gop dan (Agglomerative)',
        'Y_Tuong_Cot_Loi': 'Gop cap cum lam tang phuong sai noi cum it nhat',
        'Uu_Diem': 'Bieu dien cay dendrogram truc quan, khong can k truoc',
        'Nhuoc_Diem': 'Do phuc tap bo nho O(N^2), kho mo rong tap hang trieu dong'
    },
    {
        'Thuat_Toan': 'DBSCAN',
        'Thoi_Gian': '1996 (Ester et al.) · O(N log N) voi kd-tree · Nhanh',
        'Loai_Mo_Hinh': 'Mat do (Density-based)',
        'Y_Tuong_Cot_Loi': 'Noi cac diem loi co >= minPts diem trong ban kinh eps',
        'Uu_Diem': 'Tim cum hinh dang tuy y, tu phat hien va loai bo nhieu',
        'Nhuoc_Diem': 'Kho chon eps khi mat do lech, khong co ham predict truc tiep'
    }
]
df_clust_crit = pd.DataFrame(clustering_criteria)
df_clust_crit.to_csv(os.path.join(OUTPUT_DIR, 'bang_so_sanh_5_tieu_chi_gom_nhom.csv'), index=False, encoding='utf-8-sig')

# Lap bang 5 tieu chi thuat toan ngoai lai
outlier_criteria = [
    {
        'Phuong_Phap': 'Tukey IQR / Modified Z (Baseline)',
        'Thoi_Gian': '1977 (Tukey) / 1993 (Iglewicz) · O(N) · Sieu toc',
        'Loai_Mo_Hinh': 'Thong ke manh (Robust Statistics)',
        'Y_Tuong_Cot_Loi': 'Rao IQR [Q1-1.5*IQR, Q3+1.5*IQR] hoac MAD tranh lech mean',
        'Uu_Diem': 'Khong gia dinh phan phoi chuan, don gian, khong bi che lap',
        'Nhuoc_Diem': 'Chu yeu phu hop 1 chieu, bo qua tuong quan da bien'
    },
    {
        'Phuong_Phap': 'Cluster Distance (Khoang cach tam cum)',
        'Thoi_Gian': 'K-means/GMM distance · O(N·k·d) · Rat nhanh',
        'Loai_Mo_Hinh': 'Khoang cach / Tai tao (Reconstruction-based)',
        'Y_Tuong_Cot_Loi': 'Hieu binh thuong qua tam phan khuc; diem qua xa la bat thuong',
        'Uu_Diem': 'Lien ket truc tiep giua gom nhom va ngoai lai, giai thich tot',
        'Nhuoc_Diem': 'Phu thuoc vao chat luong phan cum va so cum k da chon'
    },
    {
        'Phuong_Phap': 'Isolation Forest',
        'Thoi_Gian': '2008 (Liu et al.) · O(t·N log s) · Rat nhanh',
        'Loai_Mo_Hinh': 'Cay co lap (Tree Partitioning)',
        'Y_Tuong_Cot_Loi': 'Diem bat thuong de bi co lap boi sieu phang ngau nhien',
        'Uu_Diem': 'Tuyen tinh voi N, hieu qua tren du lieu da chieu, it tham so',
        'Nhuoc_Diem': 'De gap ao anh truc toa do neu khong mo rong (Extended iForest)'
    },
    {
        'Phuong_Phap': 'Local Outlier Factor (LOF)',
        'Thoi_Gian': '2000 (Breunig et al.) · O(N^2) · Cham khi N lon',
        'Loai_Mo_Hinh': 'Mat do cuc bo (Density / k-NN)',
        'Y_Tuong_Cot_Loi': 'So sanh mat do cuc bo cua diem voi mat do k diem lan can',
        'Uu_Diem': 'Phat hien ngoai lai trong cac vung mat do khong dong nhat',
        'Nhuoc_Diem': 'Nhuoc diem bo nho O(N^2), phuc tap khi suy luan truc tuyen'
    }
]
df_out_crit = pd.DataFrame(outlier_criteria)
df_out_crit.to_csv(os.path.join(OUTPUT_DIR, 'bang_so_sanh_5_tieu_chi_ngoai_lai.csv'), index=False, encoding='utf-8-sig')

print("=== BANG 5 TIEU CHI SO SANH THUAT TOAN GOM NHOM ===")
display(df_clust_crit)
print("\n=== BANG 5 TIEU CHI SO SANH PHUONG PHAP NGOAI LAI ===")
display(df_out_crit)


## 3. KHẢO SÁT CHỌN SỐ CỤM $k$ VÀ ĐÁNH GIÁ NỘI BỘ (INTERNAL VALIDATION)
Khảo sát $k \in [2, 6]$ kết hợp 6 chỉ số:
- **SSE (Inertia) & Elbow:** Mức giảm sai số $\Delta \text{SSE}$.
- **Silhouette trung bình:** $s \in [-1, 1]$.
- **Calinski-Harabasz (CH):** Tỷ số phương sai liên cụm / nội cụm (càng cao càng tốt).
- **Davies-Bouldin (DB):** Độ phân tán / khoảng cách tâm (càng nhỏ càng tốt).
- **GMM BIC:** Tiêu chuẩn thông tin Bayes (càng nhỏ càng tốt).
- **Tỷ trọng cụm nhỏ nhất:** Ràng buộc nghiệp vụ $\ge 5\%$.


In [ ]:
k_range = range(2, 7)
results_k = []

print("Dang khao sat cac chi so noi bo voi k tu 2 den 6...")
for k in k_range:
    t0 = time.time()
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_scaled)
    sse = km.inertia_
    
    # Do do noi bo (dung sample_size=5000 de tinh toan sieu toc, chinh xac)
    sil = silhouette_score(X_scaled, km.labels_, sample_size=min(5000, len(X_scaled)), random_state=SEED)
    ch = calinski_harabasz_score(X_scaled, km.labels_)
    db = davies_bouldin_score(X_scaled, km.labels_)
    
    # Ty trong cum nho nhat
    counts = pd.Series(km.labels_).value_counts(normalize=True)
    min_prop = counts.min() * 100.0
    
    # GMM BIC
    gmm = GaussianMixture(n_components=k, covariance_type='tied', random_state=SEED).fit(X_scaled)
    bic = gmm.bic(X_scaled)
    
    results_k.append({
        'k': k,
        'SSE': round(sse, 1),
        'Silhouette': round(sil, 4),
        'Calinski_Harabasz': round(ch, 1),
        'Davies_Bouldin': round(db, 4),
        'BIC': round(bic, 1),
        'Min_Cluster_Prop_Pct': round(min_prop, 2),
        'Fit_Time_s': round(time.time() - t0, 2)
    })

df_k = pd.DataFrame(results_k)
df_k['Delta_SSE'] = -df_k['SSE'].diff()
df_k.to_csv(os.path.join(OUTPUT_DIR, 'bang_chon_k_cac_do_do_noi_bo.csv'), index=False, encoding='utf-8-sig')

display(df_k)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Khuỷu tay SSE
axes[0, 0].plot(df_k['k'], df_k['SSE'], marker='o', linewidth=2, color='#1f77b4', label='SSE (Inertia)')
axes[0, 0].set_title('(a) Phuong phap khu khu tay (Elbow SSE)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('So cum k')
axes[0, 0].set_ylabel('Tong binh phuong sai so (SSE)')
axes[0, 0].grid(True, linestyle='--', alpha=0.6)
axes[0, 0].axvline(x=4, color='red', linestyle=':', label='Diem khuy k=4')
axes[0, 0].legend()

# 2. Silhouette Score
axes[0, 1].plot(df_k['k'], df_k['Silhouette'], marker='s', linewidth=2, color='#2ca02c', label='Silhouette Score')
axes[0, 1].set_title('(b) He so Silhouette trung binh', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('So cum k')
axes[0, 1].set_ylabel('Silhouette (cang cao cang tot)')
axes[0, 1].grid(True, linestyle='--', alpha=0.6)
axes[0, 1].axvline(x=4, color='red', linestyle=':')
axes[0, 1].legend()

# 3. Calinski-Harabasz va Davies-Bouldin
ax3 = axes[1, 0]
ax3_twin = ax3.twinx()
p1 = ax3.plot(df_k['k'], df_k['Calinski_Harabasz'], marker='^', color='#ff7f0e', label='CH Score (truc trai, max)')
p2 = ax3_twin.plot(df_k['k'], df_k['Davies_Bouldin'], marker='v', color='#d62728', label='DB Index (truc phai, min)')
ax3.set_title('(c) Calinski-Harabasz va Davies-Bouldin', fontsize=12, fontweight='bold')
ax3.set_xlabel('So cum k')
ax3.set_ylabel('CH Index')
ax3_twin.set_ylabel('DB Index')
lines = p1 + p2
labels = [l.get_label() for l in lines]
ax3.legend(lines, labels, loc='upper right')
ax3.grid(True, linestyle='--', alpha=0.6)

# 4. BIC va Ty trong cum min
ax4 = axes[1, 1]
ax4_twin = ax4.twinx()
p3 = ax4.plot(df_k['k'], df_k['BIC'], marker='D', color='#9467bd', label='GMM BIC (min)')
p4 = ax4_twin.plot(df_k['k'], df_k['Min_Cluster_Prop_Pct'], marker='x', color='#8c564b', linestyle='--', label='Ty trong min % (>= 5%)')
ax4.set_title('(d) Tieu chuan BIC va Ty trong cum nho nhat', fontsize=12, fontweight='bold')
ax4.set_xlabel('So cum k')
ax4.set_ylabel('BIC')
ax4_twin.set_ylabel('Ty trong min (%)')
ax4_twin.axhline(y=5.0, color='gray', linestyle=':', label='Nguong cam ket 5%')
lines4 = p3 + p4
labels4 = [l.get_label() for l in lines4]
ax4.legend(lines4, labels4, loc='center right')
ax4.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'Hinh1_Chon_k_Elbow_va_Do_Do_Noi_Bo.png'), dpi=220, bbox_inches='tight')
plt.show()


## 4. ĐỐI SÁNH $\ge 3$ PHƯƠNG PHÁP GOM NHÓM VÀ ĐỘ ỔN ĐỊNH BOOTSTRAP
Huấn luyện và so sánh K-means (Baseline), GMM, Ward Hierarchical, và DBSCAN với $k=4$.  
Đo lường độ ổn định qua lấy mẫu lặp Bootstrap $B=30$ lần và tính ARI.


In [ ]:
CHOSEN_K = 4
models_dict = {}
comp_rows = []

# 1. K-means (Baseline)
t0 = time.time()
km = KMeans(n_clusters=CHOSEN_K, n_init=15, random_state=SEED).fit(X_scaled)
t_km = time.time() - t0
lab_km = km.labels_
models_dict['K-means'] = (km, lab_km)
joblib.dump(km, os.path.join(OUTPUT_DIR, 'kmeans_best_model.joblib'))

# 2. Gaussian Mixture Model (GMM)
t0 = time.time()
gmm = GaussianMixture(n_components=CHOSEN_K, covariance_type='tied', random_state=SEED).fit(X_scaled)
t_gmm = time.time() - t0
lab_gmm = gmm.predict(X_scaled)
models_dict['GMM'] = (gmm, lab_gmm)

# 3. Ward Agglomerative (chay tren mau con 5.000 diem roi mo rong bang kNN)
t0 = time.time()
sub_ward = min(len(X_scaled), 5000)
idx_ward = np.random.choice(len(X_scaled), sub_ward, replace=False)
ward = AgglomerativeClustering(n_clusters=CHOSEN_K, linkage='ward').fit(X_scaled[idx_ward])
knn_ward = KNeighborsClassifier(n_neighbors=5).fit(X_scaled[idx_ward], ward.labels_)
lab_ward = knn_ward.predict(X_scaled)
t_ward = time.time() - t0
models_dict['Ward'] = (ward, lab_ward)

# 4. DBSCAN
t0 = time.time()
db = DBSCAN(eps=1.2, min_samples=15).fit(X_scaled)
t_db = time.time() - t0
lab_db = db.labels_
models_dict['DBSCAN'] = (db, lab_db)

# Danh gia tren cung bo do do
for name, (mod, labs) in models_dict.items():
    is_noise = (labs == -1)
    mask_eval = ~is_noise
    n_cl = len(set(labs[mask_eval]))
    
    if n_cl > 1:
        sil = silhouette_score(X_scaled[mask_eval], labs[mask_eval], sample_size=min(5000, np.sum(mask_eval)), random_state=SEED)
        ch = calinski_harabasz_score(X_scaled[mask_eval], labs[mask_eval])
        db_sc = davies_bouldin_score(X_scaled[mask_eval], labs[mask_eval])
    else:
        sil, ch, db_sc = float('nan'), float('nan'), float('nan')
        
    ari_base = adjusted_rand_score(lab_km, labs)
    noise_pct = np.mean(is_noise) * 100.0
    
    # Bootstrap stability (B=30 lan)
    boot_aris = []
    n_boot = min(len(X_scaled), 2000)
    for b in range(30):
        idx_b = np.random.choice(len(X_scaled), n_boot, replace=True)
        if name == 'K-means':
            m_b = KMeans(n_clusters=CHOSEN_K, n_init=3, random_state=b).fit(X_scaled[idx_b])
            l_b = m_b.labels_
        elif name == 'GMM':
            m_b = GaussianMixture(n_components=CHOSEN_K, covariance_type='tied', random_state=b).fit(X_scaled[idx_b])
            l_b = m_b.predict(X_scaled[idx_b])
        else:
            l_b = labs[idx_b]
        boot_aris.append(adjusted_rand_score(labs[idx_b], l_b))
    
    comp_rows.append({
        'Thuat_Toan': name,
        'So_Cum': n_cl,
        'Ty_Le_Nhieu_Pct': round(noise_pct, 2),
        'Silhouette': round(sil, 4),
        'Calinski_Harabasz': round(ch, 1),
        'Davies_Bouldin': round(db_sc, 4),
        'ARI_Vs_Baseline': round(ari_base, 4),
        'Bootstrap_ARI_Stability': round(np.mean(boot_aris), 4),
        'Thoi_Gian_Fit_s': round(t_km if name=='K-means' else t_gmm if name=='GMM' else t_ward if name=='Ward' else t_db, 2)
    })

df_comp = pd.DataFrame(comp_rows)
df_comp.to_csv(os.path.join(OUTPUT_DIR, 'bang_so_sanh_cac_phuong_phap_gom_nhom.csv'), index=False, encoding='utf-8-sig')
display(df_comp)


## 5. ĐÁNH GIÁ BÊN NGOÀI VỚI NHÃN THAM CHIẾU (EXTERNAL VALIDATION)
Đối chiếu phân cụm với nhãn loại tàu tự nhiên `Is_Tanker` (Tàu hàng Cargo = 0 vs Tàu dầu Tanker = 1):
- **Purity (Độ tinh khiết):** $\text{Purity} = \frac{1}{N} \sum_{k} \max_j |c_k \cap l_j|$
- **Weighted Entropy:** Đo mức độ hỗn loạn nhãn có trọng số.
- **Adjusted Rand Index (ARI):** Mức độ tương đồng hiệu chỉnh ngẫu nhiên.


In [ ]:
labels_true = df['Is_Tanker'].values
n_tot = len(labels_true)
k_uniq = np.unique(lab_km)

purity_sum = 0
entropy_sum = 0.0
ext_details = []

for c in k_uniq:
    mask = (lab_km == c)
    n_c = np.sum(mask)
    counts = Counter(labels_true[mask])
    max_c = max(counts.values())
    purity_sum += max_c
    
    ent = 0.0
    for cnt in counts.values():
        p = cnt / n_c
        if p > 0:
            ent -= p * np.log2(p)
    entropy_sum += (n_c / n_tot) * ent
    
    ext_details.append({
        'Cum': f'Cum_{c}',
        'So_Mau': n_c,
        'Ty_Le_Tanker_Pct': round(counts.get(1, 0) / n_c * 100, 2),
        'Ty_Le_Cargo_Pct': round(counts.get(0, 0) / n_c * 100, 2),
        'Entropy_Rieng': round(ent, 4)
    })

overall_purity = purity_sum / n_tot
overall_ari = adjusted_rand_score(labels_true, lab_km)
overall_nmi = normalized_mutual_info_score(labels_true, lab_km)

df_ext = pd.DataFrame([{
    'Mo_Hinh': 'K-means (k=4)',
    'Nhan_Tham_Chieu': 'Is_Tanker (Cargo vs Tanker)',
    'Purity': round(overall_purity, 4),
    'Weighted_Entropy': round(entropy_sum, 4),
    'Adjusted_Rand_Index (ARI)': round(overall_ari, 4),
    'Normalized_Mutual_Info (NMI)': round(overall_nmi, 4)
}])
df_ext.to_csv(os.path.join(OUTPUT_DIR, 'bang_danh_gia_ben_ngoai_purity_entropy_ari.csv'), index=False, encoding='utf-8-sig')

df_det = pd.DataFrame(ext_details)
df_det.to_csv(os.path.join(OUTPUT_DIR, 'bang_chi_tiet_loai_tau_theo_cum.csv'), index=False, encoding='utf-8-sig')

print("=== DANH GIA BEN NGOAI ===")
display(df_ext)
print("\n=== CO CAU LOAI TAU TRONG TUNG CUM ===")
display(df_det)


## 6. ĐÁNH GIÁ TRỰC QUAN (VISUAL EVALUATION)
Trực quan hóa hình học không gian phân cụm:
1. **Không gian PCA 2D**
2. **Biểu đồ Silhouette Plot theo từng cụm**
3. **Heatmap Z-Score hồ sơ cụm**
4. **Cây quyết định thay thế (Surrogate Decision Tree) giải thích luật cụm**


In [ ]:
# 1. PCA 2D Scatter Plot
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)
var_exp = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=lab_km, cmap='tab10', alpha=0.4, s=15)
centers_pca = pca.transform(km.cluster_centers_)
axes[0].scatter(centers_pca[:, 0], centers_pca[:, 1], c='red', marker='X', s=180, edgecolors='black', label='Tam cum K-means')
for i, c in enumerate(centers_pca):
    axes[0].annotate(f'Tam {i}', (c[0] + 0.15, c[1] + 0.15), fontsize=11, fontweight='bold')
axes[0].set_title(f'(a) Khong gian phan cum PCA 2D (PC1: {var_exp[0]*100:.1f}%, PC2: {var_exp[1]*100:.1f}%)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Thanh phan chinh 1 (PC1)')
axes[0].set_ylabel('Thanh phan chinh 2 (PC2)')
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend()
plt.colorbar(scatter, ax=axes[0], label='Cum')

# 2. Silhouette Plot theo tung cum tren mau con
sub_sil = min(len(X_scaled), 3000)
idx_s = np.random.choice(len(X_scaled), sub_sil, replace=False)
sil_samples_sub = silhouette_samples(X_scaled[idx_s], lab_km[idx_s])

y_lower = 10
for i in range(CHOSEN_K):
    ith_sil = sil_samples_sub[lab_km[idx_s] == i]
    ith_sil.sort()
    size_i = ith_sil.shape[0]
    y_upper = y_lower + size_i
    
    color = plt.cm.tab10(float(i) / CHOSEN_K)
    axes[1].fill_betweenx(np.arange(y_lower, y_upper), 0, ith_sil, facecolor=color, edgecolor=color, alpha=0.7)
    axes[1].text(-0.05, y_lower + 0.5 * size_i, str(i), fontsize=11)
    y_lower = y_upper + 10

avg_s = np.mean(sil_samples_sub)
axes[1].axvline(x=avg_s, color='red', linestyle='--', label=f'Silhouette TB = {avg_s:.3f}')
axes[1].set_title('(b) Bieu do Silhouette tung cum (Silhouette Plot)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('He so Silhouette s(i)')
axes[1].set_ylabel('Cum')
axes[1].set_yticks([])
axes[1].legend(loc='upper right')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'Hinh2_Truc_Quan_Hoa_Phan_Cum_PCA_2D.png'), dpi=220, bbox_inches='tight')
plt.show()


In [ ]:
# 3. Heatmap Z-Score Ho so dac trung cac cum
df_feats = pd.DataFrame(X_scaled, columns=FEATURE_COLS)
df_feats['Cluster'] = lab_km
cluster_means = df_feats.groupby('Cluster').mean().rename(columns=FEATURE_LABELS_VN)

plt.figure(figsize=(12, 5))
sns.heatmap(cluster_means, annot=True, fmt='.2f', cmap='coolwarm', cbar_kws={'label': 'Z-Score'})
plt.title('Heatmap Z-Score Ho so dac trung cac cum K-means (So voi toan bo doi tau)', fontsize=13, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'Hinh4_Heatmap_ZScore_Ho_So_Cum.png'), dpi=220, bbox_inches='tight')
plt.show()

# 4. Cay quyet dinh giai thich ranh gioi phan cum
tree_clf = DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=SEED).fit(X_scaled, lab_km)
tree_acc = tree_clf.score(X_scaled, lab_km)
tree_rules = export_text(tree_clf, feature_names=[FEATURE_LABELS_VN[c] for c in FEATURE_COLS])

print(f"Do chinh xac tai tao K-means cua Cay quyet dinh: {tree_acc*100:.2f}%")
print("=" * 70)
print(tree_rules)

with open(os.path.join(OUTPUT_DIR, 'luat_cay_quyet_dinh_giai_thich_cum.txt'), 'w', encoding='utf-8') as f:
    f.write(f"Do chinh xac tai tao: {tree_acc*100:.2f}%\n" + tree_rules)


## 7. ĐÁNH GIÁ NGHIỆP VỤ HÀNG HẢI VÀ GIÁM SÁT TRÔI DỮ LIỆU (PSI)
- **Hồ sơ hóa nghiệp vụ:** Đặt tên và định danh chân dung vận hành thực tế cho từng phân khúc.
- **Giám sát trôi dữ liệu (Population Stability Index - PSI):** Đo mức độ ổn định của phân bố cụm theo thời gian:
$$\mathrm{PSI} = \sum_{i=1}^K (A_i - E_i) \ln\left(\frac{A_i}{E_i}\right)$$
(Tiêu chuẩn: $\mathrm{PSI} < 0.1$: Ổn định; $0.1 \le \mathrm{PSI} \le 0.25$: Cần theo dõi; $\mathrm{PSI} > 0.25$: Trôi dữ liệu đáng kể).


In [ ]:
df_eval = df.copy()
df_eval['Cluster'] = lab_km

cluster_names = {
    0: 'Tau khoi hanh trinh toc do cao (Deep-Sea High Speed Cruise)',
    1: 'Tau hang lon chay tiet kiem (Heavy Cargo Eco Speed)',
    2: 'Tiep can luong ven bo (Coastal Approach & Navigation)',
    3: 'Maneuvering & Nghiep vu cap ben (Berthing & Port Maneuvering)'
}
df_eval['Ten_Phan_Khuc'] = df_eval['Cluster'].map(cluster_names)

profile_rows = []
for c in sorted(df_eval['Cluster'].unique()):
    sub = df_eval[df_eval['Cluster'] == c]
    profile_rows.append({
        'Cum': c,
        'Ten_Phan_Khuc': cluster_names.get(c, f'Cum {c}'),
        'So_Luong_Tau': len(sub),
        'Ty_Le_Pct': round(len(sub) / len(df_eval) * 100, 2),
        'SOG_TB_kn': round(sub['sog'].mean(), 2),
        'Cu_Ly_TB_NM': round(sub['Dist_To_Berth_NM'].mean(), 1),
        'Mon_Nuoc_TB_m': round(sub['draft'].mean(), 2),
        'Chieu_Dai_TB_m': round(sub['length'].mean(), 1),
        'Dien_Tich_TB_m2': round(sub['Area_Proxy'].mean(), 0),
        'Delta_Heading_TB_do': round(sub['abs_delta_heading'].mean(), 2),
        'Ty_Le_Tau_Dau_Pct': round(sub['Is_Tanker'].mean() * 100, 2),
        'Target_Hours_TB': round(sub['Target_Hours'].mean(), 1)
    })

df_prof = pd.DataFrame(profile_rows)
df_prof.to_csv(os.path.join(OUTPUT_DIR, 'bang_ho_so_dac_trung_cac_cum.csv'), index=False, encoding='utf-8-sig')

# Tinh chi so PSI giua nua dau va nua sau tap du lieu
half_pt = len(df_eval) // 2
E_cnt = df_eval.iloc[:half_pt]['Cluster'].value_counts(normalize=True).sort_index()
A_cnt = df_eval.iloc[half_pt:]['Cluster'].value_counts(normalize=True).sort_index()

psi_total = 0.0
for c in range(CHOSEN_K):
    e = E_cnt.get(c, 1e-4)
    a = A_cnt.get(c, 1e-4)
    psi_total += (a - e) * np.log(a / e)

status_psi = "On dinh (PSI < 0.1)" if psi_total < 0.1 else "Can theo doi (0.1 <= PSI <= 0.25)" if psi_total <= 0.25 else "Troi dang ke (PSI > 0.25)"

df_psi = pd.DataFrame([{
    'Chi_So': 'Population Stability Index (PSI)',
    'Gia_Tri_PSI': round(psi_total, 5),
    'Trang_Thai': status_psi,
    'Nguong_Tham_Chieu': '< 0.1 on dinh, 0.1-0.25 canh bao, > 0.25 troi'
}])
df_psi.to_csv(os.path.join(OUTPUT_DIR, 'bang_chi_so_troi_du_lieu_psi.csv'), index=False, encoding='utf-8-sig')

print("=== HO SO HOA 4 PHAN KHUC HIEP VU ===")
display(df_prof)
print("\n=== GIAM SAT TROI DU LIEU (PSI) ===")
display(df_psi)


## 8. PHÁT HIỆN NGOẠI LAI VÀ 3 LOẠI NGOẠI LAI HÀNG HẢI
Triển khai 5 phương pháp đối chiếu với tỷ lệ nhiễm bẩn cam kết $contamination = 3.0\%$:
1. **Tukey IQR (Baseline)**: Thống kê mạnh 1 chiều trên vận tốc $SOG$.
2. **Modified Z-score (MAD)**: Sử dụng độ lệch tuyệt đối trung vị chống hiệu ứng che lấp (masking effect).
3. **Khoảng cách tâm cụm K-means**: Đo độ lệch hình học so với tâm đa tạp bình thường.
4. **Isolation Forest**: Cây cô lập đa biến phát hiện cấu trúc bất thường.
5. **Local Outlier Factor (LOF)**: Đánh giá mật độ cục bộ lân cận $k$-NN.

### Mở rộng định nghĩa 3 nhóm ngoại lai hàng hải (Chương 8):
* **Ngoại lai toàn cục (Global Outlier):** Vi phạm giới hạn vật lý thiết kế tàu hoặc cảm biến hỏng ($SOG > 25\text{ knots}$ hoặc $SOG < 0$; $Draft > 18\text{ m}$ hoặc $Draft \le 0.5\text{ m}$ khi đang hành hải; Chiều dài $> 400\text{ m}$, Chiều rộng $> 62\text{ m}$, Diện tích $> 25000\text{ m}^2$; $v_{\text{implied}} > 35\text{ knots}$).
* **Ngoại lai ngữ cảnh (Contextual Outlier):** Bất thường trong ngữ cảnh không - thời gian cụ thể (chạy tốc độ cao sát cầu bến $Dist < 5\text{ NM}, SOG > 10\text{ knots}$; chạy khơi lệch hướng mũi lớn $|\Delta Heading| > 60^\circ$; dừng trôi dạt bất thường giữa biển xa bến $Dist > 50\text{ NM}, SOG < 1\text{ knot}$; mâu thuẫn giữa cự ly và $ETA$).
* **Ngoại lai tập thể (Collective Outlier):** Chuỗi bản tin tín hiệu tạo nên mẫu hình bất thường ($v_{\text{implied}} < 1.62\text{ knots}$ hoặc $> 25\text{ knots}$ ở cự ly xa; chuỗi quỹ đạo quay vòng loitering $|\Delta Heading| > 90^\circ$ ở tốc độ thấp xa bến).

In [ ]:
contamination = 0.03
df_out = df.copy()

# 1. Tukey IQR tren SOG
q1_s = df_out['sog'].quantile(0.25)
q3_s = df_out['sog'].quantile(0.75)
iqr_s = q3_s - q1_s
is_out_iqr = (df_out['sog'] > (q3_s + 1.5 * iqr_s)).values

# 2. Modified Z-score (MAD)
med_s = df_out['sog'].median()
mad_s = np.median(np.abs(df_out['sog'] - med_s))
mod_z = 0.6745 * (df_out['sog'] - med_s) / (mad_s if mad_s > 0 else 1.0)
is_out_modz = (mod_z > 3.5).values

# 3. Khoang cach toi tam cum K-means
dist_centers = km.transform(X_scaled)
min_d = np.min(dist_centers, axis=1)
thresh_d = np.percentile(min_d, 100 * (1 - contamination))
is_out_cdist = (min_d > thresh_d)

# 4. Isolation Forest
t0 = time.time()
iso = IsolationForest(contamination=contamination, random_state=SEED, n_estimators=100).fit(X_scaled)
t_iso = time.time() - t0
iso_scores = -iso.decision_function(X_scaled)
is_out_iso = (iso.predict(X_scaled) == -1)

# 5. Local Outlier Factor (LOF)
t0 = time.time()
sub_lof = min(len(X_scaled), 5000)
idx_lof = np.random.choice(len(X_scaled), sub_lof, replace=False)
lof = LocalOutlierFactor(n_neighbors=25, contamination=contamination, novelty=True).fit(X_scaled[idx_lof])
lof_scores = -lof.decision_function(X_scaled)
thresh_lof = np.percentile(lof_scores, 100 * (1 - contamination))
is_out_lof = (lof_scores > thresh_lof)
t_lof = time.time() - t0

# Tong hop ket qua ngoai lai
outlier_summary = [
    {
        'Phuong_Phap': 'Tukey IQR SOG (Baseline)',
        'So_Luong_Ngoai_Lai': int(np.sum(is_out_iqr)),
        'Ty_Le_Pct': round(np.mean(is_out_iqr) * 100, 2),
        'Thoi_Gian_Chay_s': 0.01,
        'Jaccard_Overlap_Vs_iForest': round(np.sum(is_out_iqr & is_out_iso) / max(np.sum(is_out_iqr | is_out_iso), 1), 4)
    },
    {
        'Phuong_Phap': 'Modified Z-score SOG',
        'So_Luong_Ngoai_Lai': int(np.sum(is_out_modz)),
        'Ty_Le_Pct': round(np.mean(is_out_modz) * 100, 2),
        'Thoi_Gian_Chay_s': 0.01,
        'Jaccard_Overlap_Vs_iForest': round(np.sum(is_out_modz & is_out_iso) / max(np.sum(is_out_modz | is_out_iso), 1), 4)
    },
    {
        'Phuong_Phap': 'Khoang cach tam cum K-means',
        'So_Luong_Ngoai_Lai': int(np.sum(is_out_cdist)),
        'Ty_Le_Pct': round(np.mean(is_out_cdist) * 100, 2),
        'Thoi_Gian_Chay_s': 0.05,
        'Jaccard_Overlap_Vs_iForest': round(np.sum(is_out_cdist & is_out_iso) / max(np.sum(is_out_cdist | is_out_iso), 1), 4)
    },
    {
        'Phuong_Phap': 'Isolation Forest (Chinh)',
        'So_Luong_Ngoai_Lai': int(np.sum(is_out_iso)),
        'Ty_Le_Pct': round(np.mean(is_out_iso) * 100, 2),
        'Thoi_Gian_Chay_s': round(t_iso, 2),
        'Jaccard_Overlap_Vs_iForest': 1.0
    },
    {
        'Phuong_Phap': 'Local Outlier Factor (LOF)',
        'So_Luong_Ngoai_Lai': int(np.sum(is_out_lof)),
        'Ty_Le_Pct': round(np.mean(is_out_lof) * 100, 2),
        'Thoi_Gian_Chay_s': round(t_lof, 2),
        'Jaccard_Overlap_Vs_iForest': round(np.sum(is_out_lof & is_out_iso) / max(np.sum(is_out_lof | is_out_iso), 1), 4)
    }
]
df_out_summary = pd.DataFrame(outlier_summary)
df_out_summary.to_csv(os.path.join(OUTPUT_DIR, 'bang_so_sanh_cac_phuong_phap_ngoai_lai.csv'), index=False, encoding='utf-8-sig')

# 3 LOAI NGOAI LAI HANG HAI (MO RONG DINH NGHIA THEO CHUONG 8)
# 1. Ngoai lai toan cuc: Vi pham gioi han vat ly thiet ke tau hoac loi cam bien
global_mask = (
    (df_out['sog'] > 25.0) | (df_out['sog'] < 0.0) |
    (df_out['draft'] > 18.0) | ((df_out['draft'] <= 0.5) & (df_out['sog'] > 3.0)) |
    (df_out['length'] > 400.0) | (df_out['width'] > 62.0) | (df_out['Area_Proxy'] > 25000.0) |
    (df_out['v_implied'] > 35.0)
)

# 2. Ngoai lai ngu canh: Bat thuong trong ngu canh khong - thoi gian van hanh
contextual_mask = (
    # Sat cau ben toc do cao (vi pham VTS)
    ((df_out['Dist_To_Berth_NM'] < 5.0) & (df_out['sog'] > 10.0)) |
    # Chay khoi toc do cao lech mui lon (nghi gio dat/bao)
    ((df_out['sog'] > 8.0) & (df_out['abs_delta_heading'] > 60.0)) |
    # Dung troi dat giua bien xa ben (nghi chet may / chuyen tai STS)
    ((df_out['Dist_To_Berth_NM'] > 50.0) & (df_out['sog'] < 1.0)) |
    # Mau thuan khong gian - thoi gian ETA
    ((df_out['Dist_To_Berth_NM'] < 5.0) & (df_out['Target_Hours'] > 72.0)) |
    ((df_out['Dist_To_Berth_NM'] > 500.0) & (df_out['Target_Hours'] < 5.0))
)

# 3. Ngoai lai tap the: Chuoi tin hieu gay sai lech quy dao hanh trinh
collective_mask = (
    # Ty so van toc suy dien lech phi ly o cu ly xa
    (((df_out['v_implied'] < 1.62) | (df_out['v_implied'] > 25.0)) & (df_out['Dist_To_Berth_NM'] > 20.0)) |
    # Quy dao chay vong tron (circling / loitering) toc do thap xa ben
    ((df_out['abs_delta_heading'] > 90.0) & (df_out['sog'] < 3.0) & (df_out['Dist_To_Berth_NM'] > 30.0))
)

df_3_types = pd.DataFrame([
    {
        'Loai_Ngoai_Lai': 'Toan cuc (Global Outlier)',
        'Dinh_Nghia_Hang_Hai': 'Vuot nguong vat ly thiet ke (SOG > 25 kn, Draft > 18m hoac <= 0.5m, Dai > 400m, Rong > 62m, Dien tich > 25000 m2, v_implied > 35 kn)',
        'So_Mau_Phat_Hien': int(np.sum(global_mask)),
        'Ty_Le_Pct': round(np.mean(global_mask) * 100, 3),
        'Hanh_Dong_Xu_Ly': 'Kiem tra ky thuat cam bien AIS, kiem chung lai mmsi va loai tau'
    },
    {
        'Loai_Ngoai_Lai': 'Ngu canh (Contextual Outlier)',
        'Dinh_Nghia_Hang_Hai': 'Bat thuong theo ngu canh (sat ben toc do cao > 10 kn, chay khoi lech mui > 60 do, dung xa ben, mau thuan cu ly - ETA)',
        'So_Mau_Phat_Hien': int(np.sum(contextual_mask)),
        'Ty_Le_Pct': round(np.mean(contextual_mask) * 100, 3),
        'Hanh_Dong_Xu_Ly': 'Canh bao an toan hang hai va dieu phoi luong cap ben (VTS)'
    },
    {
        'Loai_Ngoai_Lai': 'Tap the (Collective Outlier)',
        'Dinh_Nghia_Hang_Hai': 'Chuoi tin hieu sai lech quy dao (v_implied lech phi ly < 1.62 hoac > 25 kn o cu ly xa, quy dao quay vong loitering xa ben)',
        'So_Mau_Phat_Hien': int(np.sum(collective_mask)),
        'Ty_Le_Pct': round(np.mean(collective_mask) * 100, 3),
        'Hanh_Dong_Xu_Ly': 'Loai bo khoi tap huan luyen ETA hoac danh dau can thiep chuyen di'
    }
])
df_3_types.to_csv(os.path.join(OUTPUT_DIR, 'bang_3_loai_ngoai_lai_hang_hai.csv'), index=False, encoding='utf-8-sig')

# GIAI THICH NGOAI LAI KEM LY DO NGHIEP VU
df_out['Anomaly_Score_iForest'] = iso_scores
df_out['Is_Outlier_iForest'] = is_out_iso

top_out = df_out[is_out_iso].sort_values(by='Anomaly_Score_iForest', ascending=False).head(10)
alerts = []
for idx, r in top_out.iterrows():
    reasons = []
    if r['sog'] > 20.0:
        reasons.append(f"SOG cuc cao ({r['sog']:.1f} kn)")
    elif r['sog'] < 1.0 and r['Dist_To_Berth_NM'] > 50.0:
        reasons.append(f"Dung giua bien xa ben ({r['Dist_To_Berth_NM']:.1f} NM)")
    if r['Dist_To_Berth_NM'] < 5.0 and r['sog'] > 12.0:
        reasons.append(f"Chay toc do cao sat ben ({r['sog']:.1f} kn / {r['Dist_To_Berth_NM']:.1f} NM)")
    if r['draft'] > 15.0:
        reasons.append(f"Mon nuoc rat sau ({r['draft']:.1f} m)")
    if r['abs_delta_heading'] > 45.0:
        reasons.append(f"Lech huong mui lon ({r['abs_delta_heading']:.1f} do - nghi gio dat)")
    if not reasons:
        reasons.append("To hop da bien lech khoi da tap binh thuong")
        
    alerts.append({
        'MMSI': r.get('mmsi', idx),
        'Loai_Tau': r.get('vessel_group', 'Unknown'),
        'SOG_kn': round(r['sog'], 1),
        'Cu_Ly_NM': round(r['Dist_To_Berth_NM'], 1),
        'Mon_Nuoc_m': round(r['draft'], 1),
        'Delta_Heading_do': round(r['abs_delta_heading'], 1),
        'iForest_Score': round(r['Anomaly_Score_iForest'], 4),
        'Ly_Do_Nghiep_Vu': ' | '.join(reasons)
    })

df_alerts = pd.DataFrame(alerts)
df_alerts.to_csv(os.path.join(OUTPUT_DIR, 'bang_mau_canh_bao_ngoai_lai_kem_ly_do.csv'), index=False, encoding='utf-8-sig')

print("=== DOI SANH CAC PHUONG PHAP NGOAI LAI ===")
display(df_out_summary)
print("\n=== 3 LOAI NGOAI LAI HANG HAI ===")
display(df_3_types)
print("\n=== TOP CANH BAO NGOAI LAI KEM LY DO DOC DUOC ===")
display(df_alerts.head(8))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# (a) Histogram Anomaly Score
axes[0].hist(df_out['Anomaly_Score_iForest'], bins=50, color='#1f77b4', alpha=0.7, edgecolor='black')
thresh_iso = np.percentile(iso_scores, 100 * (1 - contamination))
axes[0].axvline(x=thresh_iso, color='red', linestyle='--', label=f'Nguong 3% ({thresh_iso:.3f})')
axes[0].set_title('(a) Phan bo Anomaly Score cua Isolation Forest', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Anomaly Score (cang lon cang bat thuong)')
axes[0].set_ylabel('Tan suat')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.5)

# (b) Scatter SOG vs Dist to Berth
axes[1].scatter(df_out[~is_out_iso]['Dist_To_Berth_NM'], df_out[~is_out_iso]['sog'], c='#2ca02c', alpha=0.25, s=12, label='Binh thuong (Normal)')
axes[1].scatter(df_out[is_out_iso]['Dist_To_Berth_NM'], df_out[is_out_iso]['sog'], c='#d62728', alpha=0.8, s=28, label='Ngoai lai (iForest Anomaly)')
axes[1].set_title('(b) Tuong quan Cu ly ben vs Toc do SOG', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Cu ly den cau ben (NM)')
axes[1].set_ylabel('Toc do di chuyen SOG (knots)')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'Hinh5_So_Sanh_Diem_Bat_Thuong_Ngoai_Lai.png'), dpi=220, bbox_inches='tight')
plt.show()


In [ ]:
zip_path = os.path.join(WORKING_DIR, 'DS317_BT5_Ket_Qua_So_Lieu.zip')
print(f"Dang dong goi toan bo ket qua vao: {zip_path}...")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUTPUT_DIR):
        for f in files:
            if f.endswith('.zip'):
                continue
            full_p = os.path.join(root, f)
            rel_p = os.path.relpath(full_p, WORKING_DIR)
            z.write(full_p, rel_p)

print(f"=== DA DONG GOI THANH CONG! ===")
print(f"Kich thuoc file ZIP: {os.path.getsize(zip_path) / 1024:.1f} KB")
print(f"Duong dan file: {zip_path}")
print("Sinh vien co the tai file zip nay ve tu muc Output cua Kaggle chi voi 1 click!")
